# Modelagem - Classificação de Atrasos de Voos

## Objetivo
Desenvolver um modelo de classificação para prever se um voo vai atrasar ou não, utilizando:
- **PySpark ML (MLlib)** para processamento distribuído e escalável
- **MLflow** para tracking de experimentos
- **Múltiplos algoritmos** (Random Forest, Gradient Boosted Trees, Logistic Regression)
- **Métricas robustas** para avaliação (Accuracy, Precision, Recall, F1-Score, ROC-AUC)

## Pipeline
1. Carregamento e preparação dos dados (Spark DataFrame)
2. Feature engineering e preprocessamento (Spark ML)
3. Split dos dados (train/validation/test)
4. Treinamento com otimização de hiperparâmetros
5. Comparação de modelos
6. Registro do melhor modelo no MLflow

In [0]:
# Imports essenciais
import mlflow
import mlflow.spark
import warnings
warnings.filterwarnings('ignore')

# PySpark SQL e funções
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, DoubleType

# PySpark ML - Feature Engineering
from pyspark.ml.feature import (
    VectorAssembler, StandardScaler, StringIndexer, 
    OneHotEncoder, Imputer
)

# PySpark ML - Modelos
from pyspark.ml.classification import (
    RandomForestClassifier, 
    GBTClassifier,
    LogisticRegression
)

# PySpark ML - Avaliação
from pyspark.ml.evaluation import (
    BinaryClassificationEvaluator,
    MulticlassClassificationEvaluator
)

# PySpark ML - Pipeline
from pyspark.ml import Pipeline
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator

print("✓ Bibliotecas PySpark ML importadas com sucesso!")

## Carregamento dos Dados
Carregue o dataset de voos utilizado para análise de EDA. Adapte o path conforme necessário para o workspace.

In [0]:
# Carregar dataset de voos do Unity Catalog (mantém como Spark DataFrame)
df = spark.table("workspace.postech_tech_challenge_f3.flights")

print(f'Total de registros: {df.count():,}')
print(f'Número de colunas: {len(df.columns)}')
print('\nSchema:')
df.printSchema()
print('\nPrimeiras linhas:')
display(df.limit(5))

In [0]:
# Construir coluna target: delayed (1=sim, 0=não) usando PySpark
df = df.withColumn('delayed', F.when(F.col('DEPARTURE_DELAY') > 0, 1).otherwise(0))

# Verificar distribuição do target
print('Distribuição de atraso:')
df.groupBy('delayed').count().withColumn(
    'proportion', F.col('count') / df.count()
).orderBy('delayed').show()

In [0]:
# Pré-processamento & Feature Engineering com PySpark ML
from pyspark.sql.types import StringType, IntegerType, LongType, FloatType, DoubleType

# Identificar colunas numéricas e categóricas
numeric_cols = [field.name for field in df.schema.fields 
                if isinstance(field.dataType, (IntegerType, LongType, FloatType, DoubleType))
                and field.name not in ['delayed', 'DEPARTURE_DELAY']]

categorical_cols = [field.name for field in df.schema.fields 
                    if isinstance(field.dataType, StringType)]

print(f'Colunas numéricas ({len(numeric_cols)}): {numeric_cols[:10]}...')
print(f'Colunas categóricas ({len(categorical_cols)}): {categorical_cols}')

# Remover registros com target nulo
df = df.filter(F.col('delayed').isNotNull())

# Análise de valores nulos
print('\nValores nulos por coluna (primeiras 10):')  
null_counts = df.select([F.sum(F.col(c).isNull().cast('int')).alias(c) for c in df.columns[:10]])
display(null_counts)

In [0]:
# Split dos dados usando randomSplit do Spark (70% train, 15% val, 15% test)
train_df, val_df, test_df = df.randomSplit([0.7, 0.15, 0.15], seed=42)

print(f'Train: {train_df.count():,} registros')
print(f'Validation: {val_df.count():,} registros')
print(f'Test: {test_df.count():,} registros')

# Verificar distribuição do target em cada conjunto
print('\nDistribuição do target (Train):')
train_df.groupBy('delayed').count().show()
print('Distribuição do target (Validation):')
val_df.groupBy('delayed').count().show()
print('Distribuição do target (Test):')
test_df.groupBy('delayed').count().show()

## Pipeline de Feature Engineering
Construção do pipeline Spark ML para preparar os dados:
1. **Imputação de valores ausentes** (numéricas com média)
2. **Indexação de categóricas** (StringIndexer)
3. **Vetorização de features** (VectorAssembler)
4. **Normalização** (StandardScaler)

In [0]:
# Seleção de features relevantes (removendo colunas com muitos nulos ou não relevantes)
# Remover TAIL_NUMBER (14k nulos), CANCELLATION_REASON (só relevante para voos cancelados)
features_to_use = [col for col in numeric_cols if col not in ['TAIL_NUMBER']]
cat_features_to_use = [col for col in categorical_cols if col not in ['TAIL_NUMBER', 'CANCELLATION_REASON']]

print(f'Features numéricas selecionadas ({len(features_to_use)}): {features_to_use}')
print(f'Features categóricas selecionadas ({len(cat_features_to_use)}): {cat_features_to_use}')

# Pipeline stages
stages = []

# 1. Imputação de valores ausentes em features numéricas
imputer = Imputer(
    inputCols=features_to_use,
    outputCols=[f"{c}_imputed" for c in features_to_use],
    strategy='mean'
)
stages.append(imputer)

# 2. StringIndexer para features categóricas
indexers = []
for cat_col in cat_features_to_use:
    indexer = StringIndexer(
        inputCol=cat_col, 
        outputCol=f"{cat_col}_indexed",
        handleInvalid='keep'
    )
    stages.append(indexer)
    indexers.append(f"{cat_col}_indexed")

# 3. VectorAssembler - combinar todas as features
imputed_cols = [f"{c}_imputed" for c in features_to_use]
assembler_inputs = imputed_cols + indexers
assembler = VectorAssembler(
    inputCols=assembler_inputs,
    outputCol="features_raw"
)
stages.append(assembler)

# 4. StandardScaler - normalizar features
scaler = StandardScaler(
    inputCol="features_raw",
    outputCol="features",
    withStd=True,
    withMean=False  # withMean=False para sparse vectors
)
stages.append(scaler)

print(f'\nPipeline construído com {len(stages)} stages')

## Configuração do MLflow
Configurar experimento MLflow para tracking de modelos e métricas.

In [0]:
# Configurar experimento MLflow
experiment_name = "/Workspace/Users/luizpedro6@gmail.com/experiments/flights/flight-delay-classification"
mlflow.set_experiment(experiment_name)

print(f"Experimento MLflow configurado: {experiment_name}")

## Treinamento de Modelos
Treinar e comparar múltiplos algoritmos de classificação:
- Random Forest Classifier
- Gradient Boosted Trees (GBT)
- Logistic Regression

In [0]:
# Função para treinar e avaliar modelos
def train_and_evaluate_model(model, model_name, train_data, val_data, test_data, feature_pipeline_stages):
    """
    Treina um modelo com o pipeline de features e avalia no conjunto de validação e teste.
    """
    with mlflow.start_run(run_name=model_name):
        # Log de parâmetros
        mlflow.log_param("model_type", model_name)
        mlflow.log_param("train_size", train_data.count())
        mlflow.log_param("val_size", val_data.count())
        mlflow.log_param("test_size", test_data.count())
        
        # Criar pipeline completo (feature engineering + modelo)
        pipeline = Pipeline(stages=feature_pipeline_stages + [model])
        
        # Treinar modelo
        print(f"\n{'='*60}")
        print(f"Treinando modelo: {model_name}")
        print(f"{'='*60}")
        
        pipeline_model = pipeline.fit(train_data)
        
        # Predições
        train_predictions = pipeline_model.transform(train_data)
        val_predictions = pipeline_model.transform(val_data)
        test_predictions = pipeline_model.transform(test_data)
        
        # Avaliadores
        binary_evaluator = BinaryClassificationEvaluator(labelCol="delayed", metricName="areaUnderROC")
        multiclass_evaluator = MulticlassClassificationEvaluator(labelCol="delayed")
        
        # Métricas - Train
        train_auc = binary_evaluator.evaluate(train_predictions)
        train_accuracy = multiclass_evaluator.evaluate(train_predictions, {multiclass_evaluator.metricName: "accuracy"})
        train_f1 = multiclass_evaluator.evaluate(train_predictions, {multiclass_evaluator.metricName: "f1"})
        
        # Métricas - Validation
        val_auc = binary_evaluator.evaluate(val_predictions)
        val_accuracy = multiclass_evaluator.evaluate(val_predictions, {multiclass_evaluator.metricName: "accuracy"})
        val_f1 = multiclass_evaluator.evaluate(val_predictions, {multiclass_evaluator.metricName: "f1"})
        val_precision = multiclass_evaluator.evaluate(val_predictions, {multiclass_evaluator.metricName: "weightedPrecision"})
        val_recall = multiclass_evaluator.evaluate(val_predictions, {multiclass_evaluator.metricName: "weightedRecall"})
        
        # Métricas - Test
        test_auc = binary_evaluator.evaluate(test_predictions)
        test_accuracy = multiclass_evaluator.evaluate(test_predictions, {multiclass_evaluator.metricName: "accuracy"})
        test_f1 = multiclass_evaluator.evaluate(test_predictions, {multiclass_evaluator.metricName: "f1"})
        test_precision = multiclass_evaluator.evaluate(test_predictions, {multiclass_evaluator.metricName: "weightedPrecision"})
        test_recall = multiclass_evaluator.evaluate(test_predictions, {multiclass_evaluator.metricName: "weightedRecall"})
        
        # Log de métricas no MLflow
        mlflow.log_metric("train_auc", train_auc)
        mlflow.log_metric("train_accuracy", train_accuracy)
        mlflow.log_metric("train_f1", train_f1)
        
        mlflow.log_metric("val_auc", val_auc)
        mlflow.log_metric("val_accuracy", val_accuracy)
        mlflow.log_metric("val_f1", val_f1)
        mlflow.log_metric("val_precision", val_precision)
        mlflow.log_metric("val_recall", val_recall)
        
        mlflow.log_metric("test_auc", test_auc)
        mlflow.log_metric("test_accuracy", test_accuracy)
        mlflow.log_metric("test_f1", test_f1)
        mlflow.log_metric("test_precision", test_precision)
        mlflow.log_metric("test_recall", test_recall)
        
        # Log do modelo
        mlflow.spark.log_model(pipeline_model, "model")
        
        # Imprimir resultados
        print(f"\nResultados - {model_name}:")
        print(f"\nTrain:")
        print(f"  AUC-ROC: {train_auc:.4f}")
        print(f"  Accuracy: {train_accuracy:.4f}")
        print(f"  F1-Score: {train_f1:.4f}")
        
        print(f"\nValidation:")
        print(f"  AUC-ROC: {val_auc:.4f}")
        print(f"  Accuracy: {val_accuracy:.4f}")
        print(f"  Precision: {val_precision:.4f}")
        print(f"  Recall: {val_recall:.4f}")
        print(f"  F1-Score: {val_f1:.4f}")
        
        print(f"\nTest:")
        print(f"  AUC-ROC: {test_auc:.4f}")
        print(f"  Accuracy: {test_accuracy:.4f}")
        print(f"  Precision: {test_precision:.4f}")
        print(f"  Recall: {test_recall:.4f}")
        print(f"  F1-Score: {test_f1:.4f}")
        
        return {
            'model_name': model_name,
            'val_auc': val_auc,
            'val_accuracy': val_accuracy,
            'val_f1': val_f1,
            'test_auc': test_auc,
            'test_accuracy': test_accuracy,
            'test_f1': test_f1
        }

print("Função de treinamento e avaliação criada!")

In [0]:
import os
os.environ["MLFLOW_DFS_TMP"] = "/Volumes/workspace/postech_tech_challenge_f3/mlflow_dfs_temp"

In [0]:
# Modelo 1: Random Forest Classifier
rf = RandomForestClassifier(
    labelCol="delayed",
    featuresCol="features",
    numTrees=100,
    maxDepth=10,
    seed=42
)

rf_results = train_and_evaluate_model(
    model=rf,
    model_name="Random Forest",
    train_data=train_df,
    val_data=val_df,
    test_data=test_df,
    feature_pipeline_stages=stages
)